# Phase 1 — GPU port of the 1D Euler solver (Sod shock tube)

Run this notebook on a GPU runtime: **Runtime -> Change runtime type -> T4 GPU** (free tier is enough).

This picks up exactly where the CPU version (`sod_shock_tube.cpp`, Phase 0) left off. Same physics,
same finite-volume method, same Rusanov flux. The only thing that changes is *how* the per-cell
update loop is executed: sequentially on the CPU vs. one CUDA thread per cell on the GPU.

No local Windows toolchain needed — `nvcc` is already installed and working on this machine.

In [ ]:
!nvidia-smi

## Kernel design

Each grid cell's update only depends on itself and its two immediate neighbours (a 3-point stencil) —
that's an embarrassingly parallel operation, so the mapping is direct: **one CUDA thread per cell**,
same pattern as the `vectorAdd` example, plus neighbour reads.

Two things worth calling out before you read the code:

- **Double buffering.** The kernel writes to `U_new`, never back into `U`. If we updated `U[i]` in
  place, the thread handling cell `i+1` might read the *already-updated* `U[i]` instead of the old
  value — a race condition, because there's no ordering guarantee between threads. We avoid it by
  writing to a second buffer and swapping the two pointers after every step.
- **The `dt` bottleneck.** The timestep depends on a global maximum wave speed across the whole
  domain (the CFL condition). The simplest correct way to compute that on a first GPU port is to copy
  the array back to the host and reduce it there — which means a host↔device round trip every single
  step. That's a real performance cost, and the natural next optimization (not done here) is a
  GPU-side reduction so the loop never leaves the device.

In [ ]:
%%writefile euler_gpu.cu
// ============================================================================
// 1D Euler equations solver - Phase 1: CUDA GPU port
// Same physics/numerics as the CPU version (sod_shock_tube.cpp):
// Finite Volume Method, Rusanov flux, Sod shock tube validation case.
//
// Parallelization strategy: one CUDA thread per grid cell.
// Each thread reads its own cell and its two neighbors (a 3-point stencil),
// computes the two interface fluxes, and writes the updated state.
// ============================================================================

#include <cstdio>
#include <cmath>
#include <cstdlib>
#include <chrono>
#include <fstream>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while (0)

const double GAMMA = 1.4;

struct State { double rho, mom, E; };

__device__ __host__ inline double velocity(const State& s) { return s.mom / s.rho; }

__device__ __host__ inline double pressure(const State& s) {
    double u = velocity(s);
    return (GAMMA - 1.0) * (s.E - 0.5 * s.rho * u * u);
}

__device__ __host__ inline double sound_speed(const State& s) {
    double p = pressure(s);
    return sqrt(GAMMA * p / s.rho);
}

__device__ __host__ inline State physical_flux(const State& s) {
    double u = velocity(s);
    double p = pressure(s);
    State f;
    f.rho = s.rho * u;
    f.mom = s.rho * u * u + p;
    f.E   = u * (s.E + p);
    return f;
}

__device__ __host__ inline State rusanov_flux(const State& UL, const State& UR) {
    State FL = physical_flux(UL);
    State FR = physical_flux(UR);
    double sL = fabs(velocity(UL)) + sound_speed(UL);
    double sR = fabs(velocity(UR)) + sound_speed(UR);
    double Smax = fmax(sL, sR);
    State F;
    F.rho = 0.5 * (FL.rho + FR.rho) - 0.5 * Smax * (UR.rho - UL.rho);
    F.mom = 0.5 * (FL.mom + FR.mom) - 0.5 * Smax * (UR.mom - UL.mom);
    F.E   = 0.5 * (FL.E   + FR.E)   - 0.5 * Smax * (UR.E   - UL.E);
    return F;
}

// Transmissive (outflow) boundary condition on the ghost cells.
// A 1-thread kernel is wasteful in principle, but its cost is negligible
// next to the main update kernel at this problem size.
__global__ void apply_bc_kernel(State* U, int N) {
    U[0]     = U[1];
    U[N + 1] = U[N];
}

// One thread per interior cell (i = 1..N). Writes to U_new, never to U in
// place -- see the double-buffering note above.
__global__ void update_kernel(const State* U, State* U_new, int N, double dt, double dx) {
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    if (i <= N) {
        State F_left  = rusanov_flux(U[i - 1], U[i]);
        State F_right = rusanov_flux(U[i], U[i + 1]);
        U_new[i].rho = U[i].rho - (dt / dx) * (F_right.rho - F_left.rho);
        U_new[i].mom = U[i].mom - (dt / dx) * (F_right.mom - F_left.mom);
        U_new[i].E   = U[i].E   - (dt / dx) * (F_right.E   - F_left.E);
    }
}

State primitive_to_conservative(double rho, double u, double p) {
    State s;
    s.rho = rho;
    s.mom = rho * u;
    s.E   = p / (GAMMA - 1.0) + 0.5 * rho * u * u;
    return s;
}

int main() {
    const int    N       = 400;
    const double x_min   = 0.0;
    const double x_max   = 1.0;
    const double dx      = (x_max - x_min) / N;
    const double t_final = 0.20;
    const double CFL     = 0.45;

    State* h_U = new State[N + 2];
    for (int i = 0; i < N + 2; ++i) {
        double x = x_min + (i - 0.5) * dx;
        h_U[i] = (x < 0.5) ? primitive_to_conservative(1.0, 0.0, 1.0)
                            : primitive_to_conservative(0.125, 0.0, 0.1);
    }

    State *d_U, *d_U_new;
    size_t bytes = (N + 2) * sizeof(State);
    CUDA_CHECK(cudaMalloc(&d_U, bytes));
    CUDA_CHECK(cudaMalloc(&d_U_new, bytes));
    CUDA_CHECK(cudaMemcpy(d_U, h_U, bytes, cudaMemcpyHostToDevice));

    const int THREADS = 256;
    const int BLOCKS  = (N + THREADS - 1) / THREADS;

    auto t_start = std::chrono::high_resolution_clock::now();

    double t = 0.0;
    int step = 0;
    while (t < t_final) {
        apply_bc_kernel<<<1, 1>>>(d_U, N);

        CUDA_CHECK(cudaMemcpy(h_U, d_U, bytes, cudaMemcpyDeviceToHost));
        double Smax = 0.0;
        for (int i = 1; i <= N; ++i)
            Smax = fmax(Smax, fabs(velocity(h_U[i])) + sound_speed(h_U[i]));
        double dt = CFL * dx / Smax;
        if (t + dt > t_final) dt = t_final - t;

        update_kernel<<<BLOCKS, THREADS>>>(d_U, d_U_new, N, dt, dx);
        CUDA_CHECK(cudaGetLastError());

        std::swap(d_U, d_U_new);

        t += dt;
        ++step;
    }
    CUDA_CHECK(cudaDeviceSynchronize());

    auto t_end = std::chrono::high_resolution_clock::now();
    double elapsed_ms = std::chrono::duration<double, std::milli>(t_end - t_start).count();

    CUDA_CHECK(cudaMemcpy(h_U, d_U, bytes, cudaMemcpyDeviceToHost));

    printf("GPU: %d steps, final t = %f, wall time = %.2f ms\n", step, t, elapsed_ms);

    std::ofstream out("sod_result_gpu.csv");
    out << "x,rho,u,p\n";
    for (int i = 1; i <= N; ++i) {
        double x = x_min + (i - 0.5) * dx;
        out << x << "," << h_U[i].rho << "," << velocity(h_U[i]) << "," << pressure(h_U[i]) << "\n";
    }
    out.close();
    printf("Wrote sod_result_gpu.csv\n");

    cudaFree(d_U);
    cudaFree(d_U_new);
    delete[] h_U;
    return 0;
}

In [ ]:
!nvcc -O3 euler_gpu.cu -o euler_gpu
!./euler_gpu

## CPU baseline (same code as Phase 0, with timing added) — for the speedup comparison

In [ ]:
%%writefile euler_cpu.cpp
#include <cstdio>
#include <cmath>
#include <vector>
#include <algorithm>
#include <fstream>
#include <chrono>

const double GAMMA = 1.4;

struct State { double rho, mom, E; };

double velocity(const State& s) { return s.mom / s.rho; }
double pressure(const State& s) {
    double u = velocity(s);
    return (GAMMA - 1.0) * (s.E - 0.5 * s.rho * u * u);
}
double sound_speed(const State& s) {
    double p = pressure(s);
    return std::sqrt(GAMMA * p / s.rho);
}
State physical_flux(const State& s) {
    double u = velocity(s);
    double p = pressure(s);
    State f;
    f.rho = s.rho * u;
    f.mom = s.rho * u * u + p;
    f.E   = u * (s.E + p);
    return f;
}
State rusanov_flux(const State& UL, const State& UR) {
    State FL = physical_flux(UL);
    State FR = physical_flux(UR);
    double sL = std::fabs(velocity(UL)) + sound_speed(UL);
    double sR = std::fabs(velocity(UR)) + sound_speed(UR);
    double Smax = std::max(sL, sR);
    State F;
    F.rho = 0.5 * (FL.rho + FR.rho) - 0.5 * Smax * (UR.rho - UL.rho);
    F.mom = 0.5 * (FL.mom + FR.mom) - 0.5 * Smax * (UR.mom - UL.mom);
    F.E   = 0.5 * (FL.E   + FR.E)   - 0.5 * Smax * (UR.E   - UL.E);
    return F;
}
State primitive_to_conservative(double rho, double u, double p) {
    State s;
    s.rho = rho;
    s.mom = rho * u;
    s.E   = p / (GAMMA - 1.0) + 0.5 * rho * u * u;
    return s;
}

int main() {
    const int    N        = 400;
    const double x_min    = 0.0;
    const double x_max    = 1.0;
    const double dx       = (x_max - x_min) / N;
    const double t_final  = 0.20;
    const double CFL      = 0.45;

    std::vector<State> U(N + 2);
    for (int i = 0; i < N + 2; ++i) {
        double x = x_min + (i - 0.5) * dx;
        U[i] = (x < 0.5) ? primitive_to_conservative(1.0, 0.0, 1.0)
                          : primitive_to_conservative(0.125, 0.0, 0.1);
    }

    auto t_start = std::chrono::high_resolution_clock::now();

    double t = 0.0;
    int step = 0;
    while (t < t_final) {
        U[0]     = U[1];
        U[N + 1] = U[N];

        double Smax = 0.0;
        for (int i = 1; i <= N; ++i)
            Smax = std::max(Smax, std::fabs(velocity(U[i])) + sound_speed(U[i]));
        double dt = CFL * dx / Smax;
        if (t + dt > t_final) dt = t_final - t;

        std::vector<State> F(N + 1);
        for (int i = 0; i <= N; ++i)
            F[i] = rusanov_flux(U[i], U[i + 1]);

        std::vector<State> U_new = U;
        for (int i = 1; i <= N; ++i) {
            U_new[i].rho = U[i].rho - (dt / dx) * (F[i].rho - F[i - 1].rho);
            U_new[i].mom = U[i].mom - (dt / dx) * (F[i].mom - F[i - 1].mom);
            U_new[i].E   = U[i].E   - (dt / dx) * (F[i].E   - F[i - 1].E);
        }
        U = U_new;

        t += dt;
        ++step;
    }

    auto t_end = std::chrono::high_resolution_clock::now();
    double elapsed_ms = std::chrono::duration<double, std::milli>(t_end - t_start).count();

    printf("CPU: %d steps, final t = %f, wall time = %.2f ms\n", step, t, elapsed_ms);

    std::ofstream out("sod_result_cpu.csv");
    out << "x,rho,u,p\n";
    for (int i = 1; i <= N; ++i) {
        double x = x_min + (i - 0.5) * dx;
        out << x << "," << U[i].rho << "," << velocity(U[i]) << "," << pressure(U[i]) << "\n";
    }
    out.close();
    printf("Wrote sod_result_cpu.csv\n");
    return 0;
}

In [ ]:
!g++ -O3 -std=c++17 euler_cpu.cpp -o euler_cpu
!./euler_cpu

## Compare CPU vs GPU: correctness (do the profiles match?) and speed

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cpu = pd.read_csv("sod_result_cpu.csv")
gpu = pd.read_csv("sod_result_gpu.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(axes, ["rho", "u", "p"], ["Density", "Velocity", "Pressure"]):
    ax.plot(cpu["x"], cpu[col], label="CPU", linewidth=2)
    ax.plot(gpu["x"], gpu[col], "--", label="GPU", linewidth=2)
    ax.set_xlabel("x")
    ax.set_ylabel(label)
    ax.legend()
plt.tight_layout()
plt.show()

max_diff = (cpu[["rho", "u", "p"]] - gpu[["rho", "u", "p"]]).abs().max()
print("Max |CPU - GPU| difference per field (should be ~machine precision):")
print(max_diff)

## Next steps (Phase 2, optional)

- Compare the *timing* numbers printed above at N=400 — at this tiny grid size the GPU will likely
  **not** win, because kernel launch overhead and the per-step host↔device `dt` round trip dominate.
  Try increasing `N` in both files (1000, 10000, 100000) and re-running — the crossover point where
  the GPU starts winning is itself a useful thing to report.
- Remove the per-step host↔device copy by doing the CFL reduction on the GPU (e.g. `thrust::reduce`
  or a hand-written reduction kernel).
- Extend to 2D (supersonic flow over a wedge) or add a shared-memory tiled version of the update
  kernel, tying back to the memory-coalescing discussion.